# Phase 1.7 - 1.8: Spatial Clustering with DBSCAN
Projecting coordinates and identifying spatial clusters of parking violations.

In [1]:
import pandas as pd
import numpy as np
from pyproj import Transformer
from sklearn.cluster import DBSCAN
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('exports/processed_violations.parquet')
print(f"Loaded {len(df)} rows.")


Loaded 298450 rows.


## Coordinate Projection (WGS84 -> UTM Zone 43N)

In [2]:
# WGS84 -> UTM Zone 43N (Bengaluru)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True)
x, y = transformer.transform(df['longitude'].values, df['latitude'].values)
coords_m = np.column_stack([x, y])
df['x_m'] = x
df['y_m'] = y
print("Coordinates projected successfully.")


Coordinates projected successfully.


## DBSCAN Clustering

In [3]:
# Using 80m eps and min_samples=15
db = DBSCAN(eps=80, min_samples=15, algorithm='ball_tree', metric='euclidean', n_jobs=-1)
df['cluster_id'] = db.fit_predict(coords_m)

n_clusters = len(set(df['cluster_id'])) - (1 if -1 in df['cluster_id'] else 0)
n_noise = list(df['cluster_id']).count(-1)

print(f"DBSCAN produced {n_clusters} clusters, {n_noise} noise points.")

if n_clusters > 0:
    top_cluster = df['cluster_id'].value_counts().index[0]
    if top_cluster == -1 and n_clusters > 0:
        top_cluster = df['cluster_id'].value_counts().index[1]
    top_cluster_count = (df['cluster_id'] == top_cluster).sum()
    top_cluster_loc = df[df['cluster_id'] == top_cluster]['location'].mode()[0]
    print(f"Top cluster (ID {top_cluster}): {top_cluster_loc}, {top_cluster_count} violations.")

df.to_parquet('exports/clustered_violations.parquet', index=False)
print("Saved clustered data to exports/clustered_violations.parquet")


DBSCAN produced 568 clusters, 8303 noise points.
Top cluster (ID 2): 5th Main Road, Kempe Gowda Circle, Gandhi Nagar, Bengaluru, Karnataka. Pin-560009 (India), 62282 violations.


Saved clustered data to exports/clustered_violations.parquet
